In [ ]:
# Executing this cell will disable all TQDM outputs in stdout.
import os

os.environ["DISABLE_TQDM"] = "True"

# Accessing audio data between arbitrary timestamps with the Core API [^download]

[^download]: This notebook can be downloaded as **{nb-download}`example_from_events_core.ipynb`**.

The `_static/detections/aplose_results.csv` file contains a `start_datetime` and a `end_datetime` column that represent the start and stop of time regions of interest within the audio dataset.

Our goal here is to plot a spectrogram for each of these time region using `OSEkit`'s `Core API`.

First, we will parse each time region as an `OSEkit` `Event`:

In [ ]:
from pathlib import Path
import pandas as pd
from osekit.core.event import Event

timestamps_file = Path(r"_static/detections/aplose_results.csv")

# Read the CSV and parse the timestamps as pandas Timestamps
timestamps_df = pd.read_csv(timestamps_file)
timestamps_df["start_datetime"] = pd.to_datetime(timestamps_df["start_datetime"])
timestamps_df["end_datetime"] = pd.to_datetime(timestamps_df["end_datetime"])

# Create the list of events:
events = [
    Event(begin=row.start_datetime, end=row.end_datetime)
    for row in timestamps_df.itertuples()
]

We can even do a bit of filtering, e.g. with keeping only the events within a certain duration range:

In [ ]:
from pandas import Timedelta

events = [e for e in events if e.duration > Timedelta(seconds=5)]

Now we need to parse all our audio files to `OSEkit` `AudioFile`s in order to fetch the data matching each event.

Here, the files are stored in the `_static/sample_audio/timestamped` folder and the begin timestamp of each file is encoded in a `"%y%m%d_%H%M%S"` format:

In [ ]:
from osekit.core.audio_file import AudioFile

audio_folder = Path(r"_static/sample_audio/timestamped")

audio_files = [
    AudioFile(path=file, strptime_format="%y%m%d_%H%M%S", timezone="UTC")
    for file in audio_folder.glob("*.wav")
]

We're all set for creating an `AudioDataset` from these `AudioFile` of which each `AudioData` will match the begin and end timestamp of the filtered events.

We will add an `Instrument` and a `Normalization` to that `AudioDataset` as in the other example notebooks:

In [ ]:
from osekit.core.audio_dataset import AudioDataset
from osekit.core.instrument import Instrument
from osekit.utils.audio import Normalization

ads = AudioDataset.from_events(
    events=events,
    files=audio_files,
    instrument=Instrument(end_to_end_db=150.0),
    sample_rate=24_000,
    normalization=Normalization.DC_REJECT,
)

# We can take a look at the begin and end timestamps of each data:
pd.DataFrame({"begin": ad.begin, "end": ad.end} for ad in ads.data)

If we want to export spectrograms of each data, we can use the `SpectroDataset` object (see the dedicated example notebooks for more info):

In [ ]:
from osekit.core.spectro_dataset import SpectroDataset
from scipy.signal import ShortTimeFFT
from scipy.signal.windows import hamming

sft = ShortTimeFFT(win=hamming(1024), hop=128, fs=ads.sample_rate)

sds = SpectroDataset.from_audio_dataset(
    audio_dataset=ads,
    fft=sft,
    v_lim=(0.0, 150.0),
)

We're now ready to export the spectrograms, but we can before that take a look at the first one to have an idea of what they will look like:

In [ ]:
import matplotlib.pyplot as plt

sds.data[0].plot()
plt.show()

We can now actually export all the spectrograms:

In [ ]:
# Export all spectrograms
sds.save_spectrogram(folder=audio_folder / "spectrograms")